# Example usage with trained model

This notebook demonstrates how to query a single subject's EHR data, preprocess it into token format, apply a cutoff date, and generate future clinical event predictions with the foundation model.

In [ ]:
# Create a new kernel if needed, e.g.:
# !python -m ipykernel install --user --name=fm

In [ ]:
%env WORKSPACE_CDR=wb-affable-acorn-7941.R2024Q3R8

In [ ]:
import os

import config
import model_util
import pandas as pd
import polars as pl
import torch
from transformers import AutoTokenizer

from verily.forecast.aou_data_loader import PolarsSequenceCurator, TimeMapper
from verily.forecast.export_data import query_data

In [ ]:
USE_MOCK_DATA = False
if USE_MOCK_DATA:
    SUBJECT_ID = 1001
    MODEL_PATH = "mock_data/model/gpt_model_dummy_317M"
    TOKENIZER_PATH = "mock_data/model/gpt_model_dummy_317M_tokenizer"
    VOCAB_PATH = "mock_data/data/dummy_vocab.txt"    
    LOCAL_CSV_PATH = "mock_data/train/ehr_v1.csv"
else:
    SUBJECT_ID = 3403933
    MODEL_PATH = "nemo/checkpoints_v3/LATEST/model/consolidated"
    TOKENIZER_PATH = None
    VOCAB_PATH = "/home/jupyter/data/vocab/vocab_v0_ac7f79c.txt"    
    
MODEL_NAME = "gpt"  # "gpt" or "qwen"
CUTOFF_DATE = "2020-01-01"  # Keep events before this date (YYYY-MM-DD)
MAX_NEW_TOKENS = 50

## Load single subject data

In [ ]:
if USE_MOCK_DATA:
    # Load from local CSV (mock data or pre-exported data)
    df = pd.read_csv(LOCAL_CSV_PATH)
    df = df[df["subject_id"] == SUBJECT_ID].copy()
    assert len(df) > 0, f"Subject {SUBJECT_ID} not found in {LOCAL_CSV_PATH}"
    print(f"Loaded {len(df)} rows for subject {SUBJECT_ID} from local CSV")
else:
    # Query BigQuery for a single subject
    CDR = os.environ["WORKSPACE_CDR"]
    query = f"""
    (
        SELECT
            person_id subject_id,
            UNIX_SECONDS(condition_start_datetime) time,
            CONCAT('CDzz', condition_concept_id) code,
            NULL numeric_value,
            VISIT_OCCURRENCE_ID visit_id,
            VISIT_OCCURRENCE_CONCEPT_NAME visit_type,
        FROM `{CDR}.ds_condition_occurrence`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            UNIX_SECONDS(procedure_datetime) time,
            CONCAT('PRzz', procedure_concept_id) code,
            NULL numeric_value,
            VISIT_OCCURRENCE_ID visit_id,
            VISIT_OCCURRENCE_CONCEPT_NAME visit_type,
        FROM `{CDR}.ds_procedure_occurrence`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            UNIX_SECONDS(measurement_datetime) time,
            CONCAT('MSzz', measurement_concept_id) code,
            VALUE_AS_NUMBER numeric_value,
            VISIT_OCCURRENCE_ID visit_id,
            VISIT_OCCURRENCE_CONCEPT_NAME visit_type,
        FROM `{CDR}.ds_measurement`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            UNIX_SECONDS(drug_exposure_start_datetime) time,
            CONCAT('DRzz', drug_concept_id) code,
            QUANTITY numeric_value,
            VISIT_OCCURRENCE_ID visit_id,
            VISIT_OCCURRENCE_CONCEPT_NAME visit_type,
        FROM `{CDR}.ds_drug_exposure`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            UNIX_SECONDS(visit_start_datetime) time,
            CONCAT('VSzz', visit_concept_id) code,
            NULL numeric_value,
            NULL visit_id,
            '' visit_type,
        FROM `{CDR}.ds_visit_occurrence`
        WHERE person_id = {SUBJECT_ID}
          AND standard_concept_name IN ('Inpatient Visit', 'Emergency Room Visit')
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            NULL time,
            CONCAT('GENzz', gender_concept_id) code,
            NULL numeric_value,
            NULL visit_id,
            '' visit_type,
        FROM `{CDR}.person`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            NULL time,
            CONCAT('SEXzz', sex_at_birth_concept_id) code,
            NULL numeric_value,
            NULL visit_id,
            '' visit_type,
        FROM `{CDR}.person`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            NULL time,
            CONCAT('RACzz', race_concept_id) code,
            NULL numeric_value,
            NULL visit_id,
            '' visit_type,
        FROM `{CDR}.person`
        WHERE person_id = {SUBJECT_ID}
    )
    UNION ALL
    (
        SELECT
            person_id subject_id,
            UNIX_SECONDS(birth_datetime) time,
            'MEDSzzBIRTH' code,
            NULL numeric_value,
            NULL visit_id,
            '' visit_type,
        FROM `{CDR}.person`
        WHERE person_id = {SUBJECT_ID}
    )
    ORDER BY subject_id, time
    """
    df = query_data(query)
    print(f"Fetched {len(df)} rows for subject {SUBJECT_ID} from BigQuery")

df['date'] = pd.to_datetime(df['time'], unit='s').dt.date
df.head(20)

Apply cutoff date filter

In [ ]:
cutoff_ts = pd.Timestamp(CUTOFF_DATE).timestamp()
print(f"Cutoff date: {CUTOFF_DATE} -> Unix timestamp: {cutoff_ts}")

n_before = len(df)
# Keep rows where time < cutoff OR time is null (demographics: GENzz, SEXzz, RACzz)
df = df[(df["time"].isna()) | (df["time"] < cutoff_ts)].copy()
n_after = len(df)
print(f"Filtered {n_before} -> {n_after} rows (removed {n_before - n_after} future events)")
df

## Preprocess into token sequence

In [ ]:
df_polars = pl.from_pandas(df)

# filter_low_codes=False: a single subject can't meet frequency thresholds
curator = PolarsSequenceCurator(filter_low_codes=False)
_, sequences_txt = curator.create_text_sequences(df_polars)

sequence_str = sequences_txt[SUBJECT_ID]
tokens = sequence_str.split(" ")
print(f"Sequence length: {len(tokens)} tokens")
print(f"\nToken sequence:\n{sequence_str}")

Tokenize

In [ ]:
# Load tokenizer: try saved tokenizer first, fall back to building from vocab
if TOKENIZER_PATH and os.path.isdir(TOKENIZER_PATH):
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    print(f"Loaded tokenizer from {TOKENIZER_PATH}")
else:
    tokenizer = model_util.get_tokenizer(MODEL_NAME, vocab_path=VOCAB_PATH)
    print(f"Built tokenizer from vocab at {VOCAB_PATH}")

tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

# Encode the sequence
encoded = tokenizer(sequence_str, return_tensors="pt")
input_ids = encoded["input_ids"]
print(f"Encoded length: {input_ids.shape[1]} token IDs")

# Report any <unk> tokens
unk_id = tokenizer.unk_token_id
if unk_id is not None:
    unk_mask = (input_ids == unk_id).squeeze()
    n_unk = unk_mask.sum().item()
    if n_unk > 0:
        unk_positions = unk_mask.nonzero(as_tuple=True)[0].tolist()
        unk_tokens = [tokens[i] for i in unk_positions if i < len(tokens)]
        print(f"WARNING: {n_unk} <unk> tokens found. OOV codes: {unk_tokens}")
    else:
        print("No <unk> tokens - all codes are in the trained vocabulary.")

# Truncate to model window size from the left (keeps most recent events)
if input_ids.shape[1] > config.MODEL_WINDOW_SIZE:
    input_ids = input_ids[:, -config.MODEL_WINDOW_SIZE:]
    print(f"Truncated to {config.MODEL_WINDOW_SIZE} tokens (left truncation)")

attention_mask = torch.ones_like(input_ids)
print(f"Final input shape: {input_ids.shape}")

## Load model

In [ ]:
model = model_util.get_trained_model(MODEL_PATH, MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded on {device} | {n_params:,} parameters")

## Generate

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=1.0,
    )

generated_ids = outputs[:, input_ids.shape[1]:]
print(f"Generated {generated_ids.shape[1]} new tokens")

## Decode and display results

In [ ]:
# Decode generated tokens
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
generated_tokens = generated_text.split()

print(f"Generated text:\n{generated_text}\n")

# Parse into a human-readable timeline
time_mapper = TimeMapper()
code_prefixes = {
    "CDzz": "Condition",
    "PRzz": "Procedure",
    "MSzz": "Measurement",
    "DRzz": "Drug",
    "VSzz": "Visit",
    "GENzz": "Gender",
    "SEXzz": "Sex",
    "RACzz": "Race",
}
visit_labels = {
    "VIOP": "Outpatient",
    "VIIP": "Inpatient",
    "VIER": "Emergency",
    "VITele": "Telehealth",
    "VIPharm": "Pharmacy",
    "VIAmb": "Ambulatory",
    "VIHome": "Home",
    "VIOther": "Other",
    "VIEnd": "End-of-Visit",
}

Concept name lookup

In [ ]:
# Requires WORKSPACE_CDR env var (AoU Workbench only)
if not USE_MOCK_DATA and "WORKSPACE_CDR" in os.environ:
    from verily.forecast.export_data import create_data_dictionary

    data_dict = create_data_dictionary()
    code_to_name = dict(zip(data_dict["code"], data_dict["concept_name"]))

    print("--- Generated Codes with Concept Names ---")
    for tok in generated_tokens:
        if any(tok.startswith(p) for p in code_prefixes):
            name = code_to_name.get(tok, "(not in dictionary)")
            print(f"  {tok} -> {name}")
else:
    print("Concept name lookup skipped (requires WORKSPACE_CDR env var).")
    print("Set USE_LOCAL_CSV = False and run in AoU Workbench to enable.")